# Study 2 (reproduction) — Networked variant on restored North China Plain data

Faithful copy of `study2_regional_spatial.ipynb`, changed in only two ways:
1. reads the restored file `regional_pm25_ncp.csv` (re-pulled from OpenAQ), and
2. filters to the ORIGINAL date window 2018-11-01 -> 2020-03-31 recovered from the old script's bytecode,
   so the reproduction is as faithful as the data allows.

Purpose: (a) check whether the restored data reproduces the published spatial gain 0.844 -> 0.857, and
(b) produce the networked table with a bootstrap CI for reviewer item 3.8(d).

**Caveats baked in:** this pull has 28 stations vs the original 16, so even with the same window the denser
network may not reproduce 0.844 -> 0.857 exactly. Report whatever it gives, honestly, next to the published value.

cwd=`notebooks/`. Outputs -> `../outputs/study2_regional_ncp/`.


In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
np.random.seed(20260608)
OUT=Path("../outputs/study2_regional_ncp"); OUT.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({"figure.dpi":130,"savefig.dpi":150,"font.size":11,"axes.grid":True,"grid.alpha":0.25,
                     "axes.spines.top":False,"axes.spines.right":False})

d = pd.read_csv("../data/openaq_regional/regional_pm25_ncp.csv", parse_dates=["datetime"])
d = d[d["value"].between(0, 1500)]
# ORIGINAL date window (recovered from old .pyc): 2018-11-01 -> 2020-03-31
w0, w1 = pd.Timestamp("2018-11-01", tz="UTC"), pd.Timestamp("2020-03-31", tz="UTC")
d = d[(d.datetime >= w0) & (d.datetime <= w1)]
cnt = d.groupby("station").size(); d = d[d.station.isin(cnt[cnt > 3000].index)].copy()
d["city"] = d.station.str.split("::").str[0]
LAT = {"Handan":36.61,"Xingtai":37.07,"Shijiazhuang":38.04,"Baoding":38.87,"Tianjin":39.08,"Beijing":39.90}
LON = {"Handan":114.49,"Xingtai":114.50,"Shijiazhuang":114.51,"Baoding":115.46,"Tianjin":117.20,"Beijing":116.41}
print(f"{d.station.nunique()} stations across {d.city.nunique()} cities | "
      f"{d.datetime.min().date()} -> {d.datetime.max().date()} | {len(d):,} rows")
print(d.groupby('city').station.nunique().to_string())


/Users/sumiyasayeed/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
/Users/sumiyasayeed/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


25 stations across 6 cities | 2018-11-01 -> 2020-03-30 | 265,606 rows
city
Baoding         5
Beijing         4
Handan          4
Shijiazhuang    3
Tianjin         5
Xingtai         4


In [2]:
piv = d.pivot_table(index="datetime", columns="station", values="value", aggfunc="mean").sort_index().resample("3h").mean()
stations = list(piv.columns); city = {s: s.split("::")[0] for s in stations}
PM = piv.to_numpy(float); T, S = PM.shape
logpm = np.log1p(np.clip(PM,0,None)); MU, SD0 = np.nanmean(logpm), np.nanstd(logpm); Z = (logpm - MU) / SD0
lat = np.array([LAT[city[s]] for s in stations])
UP = [[j for j in range(S) if lat[j] < lat[si]-0.15] or [j for j in range(S) if j!=si] for si in range(S)]
upz = np.column_stack([np.nanmean(Z[:, UP[si]], axis=1) for si in range(S)])
print(f"pivot {T} x {S}  (3h steps x stations)")

K=5; LEV=np.linspace(0,1,K); SDk=1.0+0.5*LEV; LOGC=0.5*np.log(2*np.pi)
def lgv(z): return -LOGC-np.log(SDk)-0.5*((z-2.0*LEV)/SDk)**2
def run_network(spatial=True, w_sp=0.9):
    logb = np.tile(np.log(np.exp(-3*np.arange(K))/np.exp(-3*np.arange(K)).sum()+1e-12),(S,1))
    risk = np.zeros((T,S))
    for t in range(T):
        for si in range(S):
            b=np.exp(logb[si]-logb[si].max()); b/=b.sum(); el=(b*LEV).sum(); pdn=0.05 if el>0.4 else 0.15
            A=np.zeros((K,K))
            for k in range(K):
                up=0.25 if k<K-1 else 0; dn=pdn if k>0 else 0; A[k,k]=1-up-dn
                if k<K-1:A[k,k+1]=up
                if k>0:A[k,k-1]=dn
            logbp=np.log(b@A+1e-12); ll=np.zeros(K)
            z=Z[t,si]
            if not np.isnan(z): ll += 1.4*lgv(z)
            if spatial and not np.isnan(upz[t,si]): ll += w_sp*upz[t,si]*LEV
            logt=logbp+ll; logb[si]=logt-(logt.max()+np.log(np.exp(logt-logt.max()).sum()))
            risk[t,si]=(np.exp(logb[si])*LEV).sum()
    return risk
risk_single = run_network(spatial=False)
risk_net    = run_network(spatial=True)
print("networked twin computed.")


pivot 4128 x 25  (3h steps x stations)


/var/folders/hs/v3rv3f714391byj_w2dh_q980000gn/T/ipykernel_7706/4107725814.py:7: RuntimeWarning: Mean of empty slice
  upz = np.column_stack([np.nanmean(Z[:, UP[si]], axis=1) for si in range(S)])


networked twin computed.


In [3]:
TAU=115.0; H=8   # episode within next 24h (3h steps), from currently-clean steps
def auc(s,y):
    s=np.asarray(s,float);y=np.asarray(y,int);pos,neg=s[y==1],s[y==0]
    if len(pos)==0 or len(neg)==0:return float('nan')
    o=np.argsort(s);r=np.empty(len(s));r[o]=np.arange(1,len(s)+1)
    _,inv,c=np.unique(s,return_inverse=True,return_counts=True);ss=np.zeros(len(c));np.add.at(ss,inv,r);r=(ss/c)[inv]
    return float((r[y==1].sum()-len(pos)*(len(pos)+1)/2)/(len(pos)*len(neg)))
def apr(s,y):
    s=np.asarray(s,float);y=np.asarray(y,int);o=np.argsort(-s);y=y[o];tp=np.cumsum(y);fp=np.cumsum(1-y);prec=tp/(tp+fp);rec=tp/max(y.sum(),1)
    return float(np.sum((rec-np.r_[0,rec[:-1]])*prec))
mo = piv.index.month.values
rows=[]
for si in range(S):
    pm=PM[:,si]
    for t in range(3,T-H):
        if np.isnan(pm[t]) or pm[t]>=TAU: continue
        fut=pm[t+1:t+1+H]
        if np.all(np.isnan(fut)): continue
        rows.append((risk_single[t,si], risk_net[t,si], int(np.nanmax(fut)>=TAU), mo[t]%2))
R=np.array(rows); rs,rn,y,parity = R[:,0],R[:,1],R[:,2].astype(int),R[:,3].astype(int)
te = parity==1
auc_s, auc_n = auc(rs[te],y[te]), auc(rn[te],y[te])
apr_s, apr_n = apr(rs[te],y[te]), apr(rn[te],y[te])
print(f"onset samples {len(R):,} | test {te.sum():,} | prevalence {y[te].mean():.3f}\n")
print(f"  Single-station twin        : AUROC {auc_s:.3f}  AUPRC {apr_s:.3f}")
print(f"  Networked twin (spatial)   : AUROC {auc_n:.3f}  AUPRC {apr_n:.3f}")

# bootstrap CI on the spatial gain (networked - single)
rng=np.random.default_rng(0); idx=np.where(te)[0]; gains=[]
for _ in range(2000):
    b=rng.choice(idx,len(idx),replace=True)
    if 0<y[b].sum()<len(b): gains.append(auc(rn[b],y[b])-auc(rs[b],y[b]))
gains=np.array(gains); lo,hi=np.percentile(gains,[2.5,97.5])
print(f"\n  spatial gain (networked - single): {auc_n-auc_s:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")
print(f"  (published value was 0.844 -> 0.857, gain +0.013)")

tbl=pd.DataFrame([("Single-station twin",auc_s,apr_s),("Networked twin (spatial)",auc_n,apr_n)],
                 columns=["model","AUROC","AUPRC"]).set_index("model")
tbl.to_csv(OUT/"networked_table.csv")
pd.DataFrame([{"single_auroc":auc_s,"networked_auroc":auc_n,"gain":auc_n-auc_s,"lo95":lo,"hi95":hi,
              "base_rate":float(y[te].mean()),"n_onset":int(y[te].sum()),"n_stations":S}]).to_csv(OUT/"networked_gain_ci.csv",index=False)
print("\nsaved table + CI to",OUT)


onset samples 77,700 | test 41,514 | prevalence 0.137

  Single-station twin        : AUROC 0.846  AUPRC 0.425
  Networked twin (spatial)   : AUROC 0.859  AUPRC 0.433

  spatial gain (networked - single): +0.0130  95% CI [+0.0106, +0.0154]
  (published value was 0.844 -> 0.857, gain +0.013)

saved table + CI to ../outputs/study2_regional_ncp


## Reading this for the reply

- Compare the reproduced single -> networked AUROC against the published 0.844 -> 0.857.
  - If close and the CI excludes zero: the result is reproduced and defensible; report it as the Table with CI.
  - If the gain shrinks or the CI spans zero: report honestly. Given 28 stations vs the original 16 and a possibly
    different OpenAQ pull, a non-exact match is expected; the reviewer's '...or removed' option remains available.
- If you want a closer station match to the original 16, we can trim to the densest-coverage stations per city,
  but we cannot recover exactly which 16 the original used.


## Threshold sensitivity of the networked gain (added for the revision)

115 ug/m3 is the PM2.5 concentration at which the individual air quality index of China's HJ 633-2012 reaches 150
(boundary between light and moderate pollution); 75 ug/m3 is the breakpoint for an index of 100, and 150 ug/m3 for
an index of 200. HJ 633-2012 defines these for 24-hour means; here, as in the main analysis, they are applied to
three-hour means. The filtered risks do not depend on the threshold, so only the onset labels change.
The bootstrap is the same as above (rows of the test partition, 2000 resamples, seed 0), so the 115 row must
reproduce the table above exactly; that is the check.

In [4]:
def network_at(tau):
    rows_=[]
    for si in range(S):
        pm=PM[:,si]
        for t in range(3,T-H):
            if np.isnan(pm[t]) or pm[t]>=tau: continue
            fut=pm[t+1:t+1+H]
            if np.all(np.isnan(fut)): continue
            rows_.append((risk_single[t,si], risk_net[t,si], int(np.nanmax(fut)>=tau), mo[t]%2))
    R_=np.array(rows_); rs_,rn_,y_,par_=R_[:,0],R_[:,1],R_[:,2].astype(int),R_[:,3].astype(int); te_=par_==1
    rng_=np.random.default_rng(0); idx_=np.where(te_)[0]; g_=[]
    for _ in range(2000):
        b=rng_.choice(idx_,len(idx_),replace=True)
        if 0<y_[b].sum()<len(b): g_.append(auc(rn_[b],y_[b])-auc(rs_[b],y_[b]))
    lo_,hi_=np.percentile(g_,[2.5,97.5])
    return dict(tau=tau,n_test=int(te_.sum()),n_onset=int(y_[te_].sum()),base_rate=float(y_[te_].mean()),
                single_AUROC=auc(rs_[te_],y_[te_]),networked_AUROC=auc(rn_[te_],y_[te_]),
                single_AUPRC=apr(rs_[te_],y_[te_]),networked_AUPRC=apr(rn_[te_],y_[te_]),
                gain=auc(rn_[te_],y_[te_])-auc(rs_[te_],y_[te_]),lo95=float(lo_),hi95=float(hi_))
TS=pd.DataFrame([network_at(t) for t in [75.0,115.0,150.0]])
TS.to_csv(OUT/"networked_threshold_sensitivity.csv",index=False)
print(TS.round(4).to_string(index=False))
print("\ncheck: the 115 row must equal the table above (0.846 -> 0.859, gain +0.0130 [+0.0106, +0.0154], 5688 onsets)")

  tau  n_test  n_onset  base_rate  single_AUROC  networked_AUROC  single_AUPRC  networked_AUPRC   gain   lo95   hi95
 75.0   35554     9808     0.2759        0.7617           0.7752        0.5032           0.5099 0.0136 0.0113 0.0160
115.0   41514     5688     0.1370        0.8461           0.8591        0.4252           0.4334 0.0130 0.0106 0.0154
150.0   44207     3988     0.0902        0.8850           0.8962        0.3824           0.3953 0.0112 0.0088 0.0136

check: the 115 row must equal the table above (0.846 -> 0.859, gain +0.0130 [+0.0106, +0.0154], 5688 onsets)
